## ANALSIS ESTADISTICO

## 0. CARGA DE LIBRERÍAS Y BASES DE DATOS DISPONIBLES

<BR> dataframe de salida = archivos (contiene información de todos las tablas/bd limpias diponibles con su ruta de acceso, extensión, etc.)

In [1]:
# librerias usadas
import os                    
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go
import plotly.express as px

# liberia para aplicar el test de Dickey-Fuller (ADF)
from statsmodels.tsa.stattools import adfuller

# liberia para aplicar el la correlacio cruzada (CCF)
from statsmodels.tsa.stattools import ccf
import re

# liberia para aplicar el test de Granger
from statsmodels.tsa.stattools import grangercausalitytests

In [2]:
# Ruta de la carpeta que quieres listar
carpeta = '/Users/rafafons/Library/CloudStorage/OneDrive-RafaelFonsBibiloni/01-Capacitaciones/IT Academy - Data Analyst/sprint 13 proyecto final/tablas finales'

# Listado de archivos, ignorando ocultos y ordenado por nombre (orden estable)
nombres_archivos = sorted(
    [f for f in os.listdir(carpeta) if not f.startswith('.')]
)

# Crear lista con la información de cada archivo
archivos = []
for nombre in nombres_archivos:
    ruta = os.path.join(carpeta, nombre)
    if os.path.isfile(ruta):
        info = {
            "nombre": nombre,
            "extension": os.path.splitext(nombre)[1].lower(),
            "tamaño_kb": round(os.path.getsize(ruta) / 1024, 2),
            "fecha_modificacion": pd.to_datetime(os.path.getmtime(ruta), unit='s'),
            "ruta_archivo": ruta
        }
        archivos.append(info)

# Convertir la lista en un DataFrame
archivos = pd.DataFrame(archivos).sort_values(by="nombre")

# Mostrar resultados
pd.options.display.max_colwidth = None
archivos

,nombre,extension,tamaño_kb,fecha_modificacion,ruta_archivo
0,00_Clasif_Minarquista_kpis.xlsx,.xlsx,19.02,2025-12-03 13:13:09.178107023,/Users/rafafons/Library/CloudStorage/OneDrive-RafaelFonsBibiloni/01-Capacitaciones/IT Academy - Data Analyst/sprint 13 proyecto final/tablas finales/00_Clasif_Minarquista_kpis.xlsx
1,01_lista_funciones_sub.csv,.csv,6.86,2025-12-01 10:47:27.875501633,/Users/rafafons/Library/CloudStorage/OneDrive-RafaelFonsBibiloni/01-Capacitaciones/IT Academy - Data Analyst/sprint 13 proyecto final/tablas finales/01_lista_funciones_sub.csv
2,02_gastos_totales_1995_2023.csv,.csv,1536.62,2025-12-01 10:47:31.518846750,/Users/rafafons/Library/CloudStorage/OneDrive-RafaelFonsBibiloni/01-Capacitaciones/IT Academy - Data Analyst/sprint 13 proyecto final/tablas finales/02_gastos_totales_1995_2023.csv
3,03_preocupaciones_españoles.csv,.csv,1.18,2025-11-21 09:53:43.120265722,/Users/rafafons/Library/CloudStorage/OneDrive-RafaelFonsBibiloni/01-Capacitaciones/IT Academy - Data Analyst/sprint 13 proyecto final/tablas finales/03_preocupaciones_españoles.csv
4,04_poblacion_1995_2023_esp.csv,.csv,2.02,2025-12-01 12:20:04.969184399,/Users/rafafons/Library/CloudStorage/OneDrive-RafaelFonsBibiloni/01-Capacitaciones/IT Academy - Data Analyst/sprint 13 proyecto final/tablas finales/04_poblacion_1995_2023_esp.csv
5,05_inflacion_1995_2023.csv,.csv,1.45,2025-12-01 12:20:05.167531013,/Users/rafafons/Library/CloudStorage/OneDrive-RafaelFonsBibiloni/01-Capacitaciones/IT Academy - Data Analyst/sprint 13 proyecto final/tablas finales/05_inflacion_1995_2023.csv
6,06_pib_1995_2023.csv,.csv,1.22,2025-12-01 12:20:05.271428823,/Users/rafafons/Library/CloudStorage/OneDrive-RafaelFonsBibiloni/01-Capacitaciones/IT Academy - Data Analyst/sprint 13 proyecto final/tablas finales/06_pib_1995_2023.csv
7,07_salarios_2009_2021_esp.csv,.csv,1.00,2025-12-01 12:20:05.314070463,/Users/rafafons/Library/CloudStorage/OneDrive-RafaelFonsBibiloni/01-Capacitaciones/IT Academy - Data Analyst/sprint 13 proyecto final/tablas finales/07_salarios_2009_2021_esp.csv
8,08_paro_2006_2024_esp.csv,.csv,0.20,2025-12-01 12:20:05.370635033,/Users/rafafons/Library/CloudStorage/OneDrive-RafaelFonsBibiloni/01-Capacitaciones/IT Academy - Data Analyst/sprint 13 proyecto final/tablas finales/08_paro_2006_2024_esp.csv
9,09_pobreza_2004_2024_esp.csv,.csv,0.23,2025-12-01 12:20:05.384049654,/Users/rafafons/Library/CloudStorage/OneDrive-RafaelFonsBibiloni/01-Capacitaciones/IT Academy - Data Analyst/sprint 13 proyecto final/tablas finales/09_pobreza_2004_2024_esp.csv


# 1. PREPARACIÓN DE DATOS

# 1.1 Preparación de datos: GASTOS ANUALES PUBLICOS


<BR> dataframe de salida = gastos_2023_wide (contiene gastos anuales expresados en MM € de: tipo de gasto, funciones y subfunciones de COFOG + classificacion minarquista )

In [3]:
# veo el nombre de la pestaña en forma dinamica
excel_file = pd.ExcelFile(archivos.loc[0,"ruta_archivo"])  
#importo el archivo
clasif_funciones = pd.read_excel(archivos.loc[0,"ruta_archivo"], sheet_name=excel_file.sheet_names[0])

clasif_funciones.head().columns

Index(['CodGrupoFuncional', 'Grupo funcional', 'Cod+Función',
       'CodClasificaciónFuncional', 'Subfunción', 'Cod+Subfunción',
       'Qué incluye?', 'Clasif_minarquista', 'Peso estado', 'Motivo'],
      dtype='object')

In [4]:
inflacion = pd.read_csv(archivos.loc[5,"ruta_archivo"])
inflacion.tail(5)

inflacion.isna().sum()[inflacion.isna().sum() > 0]

Series([], dtype: int64)

In [5]:
#importo el fichero
gastos = pd.read_csv(archivos.loc[2,"ruta_archivo"])
gastos.head()

# elimino vacios
gastos.dropna()

# elimino ceros
gastos = gastos[gastos["Millones Euros"]!=0]

# agrego el coeficiente de conversión
gastos_2023 = gastos.merge(inflacion[["Año", "convert_a_2023"]], on="Año", how="left")

# agrego columna 
gastos_2023["Cod+ClasifEconmica"] = gastos_2023["CodClasificaciónEconómica"].astype(str)+"-"+gastos_2023["ClasificaciónEconómica"].astype(str)

# calculo el valor actualizado
gastos_2023["Gasto(MM €)"] = round(gastos_2023["Millones Euros"] * gastos_2023["convert_a_2023"],0)

# agrego info de la tabla de clasificaion de subfunciones
gastos_2023 = gastos_2023.merge(
    clasif_funciones[['CodGrupoFuncional', 'Cod+Función','CodClasificaciónFuncional','Cod+Subfunción',
    "Clasif_minarquista"]], on="CodClasificaciónFuncional", how="left" )

# reemplo comas por puntos en las columnas que agrego:
gastos_2023["Cod+Función"] = gastos_2023["Cod+Función"].str.strip().str.replace(",", ".", regex=False)
gastos_2023["Cod+Subfunción"] = gastos_2023["Cod+Subfunción"].str.strip().str.replace(",", ".", regex=False)

# elimino columnas innecesarias
gastos_2023.drop(["Millones Euros", "convert_a_2023", "ClasificaciónEconómica"], axis=1, inplace=True)

#veirifa si hay nulos
gastos_2023.isna().sum()[gastos_2023.isna().sum() > 0]

gastos_2023.head()


,Año,CodClasificaciónEconómica,CodClasificaciónFuncional,Cod+ClasifEconmica,Gasto(MM €),CodGrupoFuncional,Cod+Función,Cod+Subfunción,Clasif_minarquista
0,1995,D.1,1.1,D.1-Remuneración de asalariados,4204.0,1.0,1-Servicios públicos generales,1.1-Órganos ejecutivos y legislativos. asuntos financieros y fiscales. asuntos exteriores,1-Esencial por definición
1,1995,P.2,1.1,P.2-Consumos intermedios,1854.0,1.0,1-Servicios públicos generales,1.1-Órganos ejecutivos y legislativos. asuntos financieros y fiscales. asuntos exteriores,1-Esencial por definición
2,1995,D.29,1.1,D.29-Otros impuestos sobre la producción,10.0,1.0,1-Servicios públicos generales,1.1-Órganos ejecutivos y legislativos. asuntos financieros y fiscales. asuntos exteriores,1-Esencial por definición
3,1995,D.7,1.1,D.7-Otras transferencias corrientes,6439.0,1.0,1-Servicios públicos generales,1.1-Órganos ejecutivos y legislativos. asuntos financieros y fiscales. asuntos exteriores,1-Esencial por definición
4,1995,P.5,1.1,P.5-Formación bruta de capital,535.0,1.0,1-Servicios públicos generales,1.1-Órganos ejecutivos y legislativos. asuntos financieros y fiscales. asuntos exteriores,1-Esencial por definición


In [6]:
gastos_2023_totales = gastos_2023.groupby(["Año"]).aggregate(Gasto_Total_Anual=("Gasto(MM €)","sum")).reset_index()
#gastos_2023_totales.info()

gastos_2023_totales.isna().sum()[gastos_2023_totales.isna().sum() > 0]

Series([], dtype: int64)

In [7]:
gastos_2023_minasquista = gastos_2023.groupby(["Año", "Clasif_minarquista"]).aggregate({"Gasto(MM €)":"sum"}).reset_index()
gastos_2023_minasquista = gastos_2023_minasquista.pivot_table(
    index="Año",
    columns="Clasif_minarquista",
    values="Gasto(MM €)",
    aggfunc="sum"
)
#gastos_2023_minasquista.info()

gastos_2023_minasquista.isna().sum()[gastos_2023_minasquista.isna().sum() > 0]

Series([], dtype: int64)

In [8]:
gastos_2023_func = gastos_2023.groupby(["Año", "Cod+Función"]).aggregate({"Gasto(MM €)":"sum"}).reset_index()
gastos_2023_func = gastos_2023_func.pivot_table(
    index="Año",
    columns="Cod+Función",
    values="Gasto(MM €)",
    aggfunc="sum"
)
#gastos_2023_func.info()

gastos_2023_func.isna().sum()[gastos_2023_func.isna().sum() > 0]

Series([], dtype: int64)

In [9]:
gastos_2023_subfunc = gastos_2023.groupby(["Año", "Cod+Subfunción"]).aggregate({"Gasto(MM €)":"sum"}).reset_index()

# elimino nulos
gastos_2023_subfunc.dropna(inplace=True)

gastos_2023_subfunc = gastos_2023_subfunc.pivot_table(
    index="Año",
    columns="Cod+Subfunción",
    values="Gasto(MM €)",
    aggfunc="sum",
    fill_value=0
)


gastos_2023_subfunc.isna().sum()[gastos_2023_subfunc.isna().sum() > 0]

#gastos_2023_subfunc.info()

nulos = gastos_2023_subfunc.isna().sum()
nulos[nulos>0]

Series([], dtype: int64)

In [10]:
gastos_2023_ClasEcon = gastos_2023.groupby(["Año", "Cod+ClasifEconmica"]).aggregate({"Gasto(MM €)":"sum"}).reset_index()
gastos_2023_ClasEcon = gastos_2023_ClasEcon.pivot_table(
    index="Año",
    columns="Cod+ClasifEconmica",
    values="Gasto(MM €)",
    aggfunc="sum"
)
gastos_2023_ClasEcon.head()

gastos_2023_ClasEcon.isna().sum()[gastos_2023_ClasEcon.isna().sum() > 0]

Series([], dtype: int64)

In [11]:
gastos_2023_wide = pd.merge(gastos_2023_totales, gastos_2023_func, on="Año")
gastos_2023_wide = gastos_2023_wide.merge(gastos_2023_subfunc, on="Año")
gastos_2023_wide = gastos_2023_wide.merge(gastos_2023_ClasEcon, on="Año")
gastos_2023_wide = gastos_2023_wide.merge(gastos_2023_minasquista, on="Año")
gastos_2023_wide.head()

#gastos_2023_wide.isna().sum()[gastos_2023_wide.isna().sum() > 0]

,Año,Gasto_Total_Anual,1-Servicios públicos generales,10-Protección social(sin pensiones),10.2-Protección social (pensiones),2-Defensa,3-Orden público y seguridad,4-Asuntos económicos,5-Protección del medio ambiente,6- Vivienda y servicios comunitarios,...,D.632-Transferencias sociales en especie: producción adquirida en el mercado,D.7-Otras transferencias corrientes,"D.9- Transferencias de capital, a pagar",NP- Adquisiciones menos cesiones de activos no financieros no producidos,P.2-Consumos intermedios,P.5-Formación bruta de capital,1-Esencial por definición,2-Esencial por adopción,3-Esencial Mínimo,4-No esencial
0,1995,392520.0,71422.0,68708.0,58792.0,12122.0,17501.0,51276.0,6975.0,9211.0,...,18384.0,12230.0,17404.0,744.0,37657.0,39132.0,55781.0,106675.0,61517.0,168547.0
1,1996,388406.0,74001.0,66935.0,60889.0,11597.0,16685.0,43943.0,7172.0,8877.0,...,18812.0,13403.0,14285.0,597.0,36903.0,33818.0,56131.0,108979.0,61566.0,161730.0
2,1997,386636.0,72111.0,66510.0,61562.0,11617.0,17080.0,42528.0,7554.0,9210.0,...,18621.0,15019.0,12099.0,1171.0,37988.0,34738.0,58857.0,109079.0,57661.0,161039.0
3,1998,400709.0,71959.0,66588.0,63465.0,11589.0,17679.0,48241.0,8144.0,10543.0,...,21351.0,16964.0,14451.0,212.0,38770.0,37794.0,63643.0,113545.0,56245.0,167276.0
4,1999,412845.0,68708.0,67780.0,66554.0,11520.0,18566.0,49605.0,9178.0,11143.0,...,23422.0,17981.0,12991.0,1270.0,40851.0,40120.0,68284.0,120999.0,52900.0,170662.0


# 1.2 agregare datos demograficos para calcular indices per capita y per PEA (pobl econ activa)

<BR> dataframe de salida = ratios_gastos_2023_wide (contiene gastos anuales expresados € per capita y € per PEA)

In [12]:
# veo el nombre de la pestaña en forma dinamica
excel_file = pd.ExcelFile(archivos.loc[12,"ruta_archivo"]) 
print("las pestañas que contiene son:", excel_file.sheet_names, "\n")

#importo el la pestaña num_3= "valores_demogr"
valores_demogr = pd.read_excel(archivos.loc[12,"ruta_archivo"], sheet_name=excel_file.sheet_names[2])
valores_demogr.info()

las pestañas que contiene son: ['indices_macro', 'valores_macro', 'valores_demogr', 'valores_asalariados', 'indices_sect_privado'] 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29 entries, 0 to 28
Data columns (total 8 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Año                           29 non-null     int64  
 1   total hab                     29 non-null     int64  
 2   cant ocupados                 29 non-null     float64
 3   cant desempleados             29 non-null     float64
 4   cant PEA                      29 non-null     int64  
 5   cant empleo público           29 non-null     int64  
 6   cant empleo privado           29 non-null     float64
 7   cant personas riesgo pobreza  29 non-null     int64  
dtypes: float64(3), int64(5)
memory usage: 1.9 KB


In [13]:
# --- PASO 1: Preparación y Alineación de Índices ---
# Versión indexada de gastos (Usando .assign y .set_index para evitar SettingWithCopyWarning):
gastos_2023_index = (
    gastos_2023_wide.copy()
    #.assign(Año=lambda df: df['Año'].astype(int))
    .set_index('Año')
)
# Versión indexada de demografía:
demogr_indexado = (
    valores_demogr.copy()
    .assign(Año=lambda df: df['Año'].astype(int))
    .set_index('Año')
)
# --- PASO 2: Obtener las Series Divisoras ---
# Series de población/PEA (limpias y indexadas)
pea = demogr_indexado['cant PEA'] / 1000000              
hab = demogr_indexado['total hab'] / 1000000 

# --- PASO 3: Creación y Concatenación de Ratios (SOLUCIÓN PERFORMANCE) ---
# 1. Creamos una lista para almacenar todas las nuevas Series de ratio.
lista_series_ratios = []

# 2. Bucle de cálculo
for col in gastos_2023_index.columns:
    serie_gasto = gastos_2023_index[col]
    # 2.1 Gasto por PEA
    serie_pea = serie_gasto.div(pea).rename(f'{col}_per_PEA')
    # 2.2 Gasto por Habitante
    serie_hab = serie_gasto.div(hab).rename(f'{col}_per_hab')

    # Añadimos las series calculadas a la lista
    lista_series_ratios.extend([serie_pea, serie_hab])
    
# 3. Concatenamos todas las Series en un solo DataFrame al final.
ratios_gastos_2023_wide = pd.concat(lista_series_ratios, axis=1)
    
# Mostrar el resultado
print(f"DataFrame de Ratios creado con {ratios_gastos_2023_wide.shape[1]} nuevas series de gasto.")
print("\n--- Cabecera del DataFrame de Ratios (Indexado por Año) ---")
ratios_gastos_2023_wide.reset_index().head()

DataFrame de Ratios creado con 188 nuevas series de gasto.

--- Cabecera del DataFrame de Ratios (Indexado por Año) ---


,Año,Gasto_Total_Anual_per_PEA,Gasto_Total_Anual_per_hab,1-Servicios públicos generales_per_PEA,1-Servicios públicos generales_per_hab,10-Protección social(sin pensiones)_per_PEA,10-Protección social(sin pensiones)_per_hab,10.2-Protección social (pensiones)_per_PEA,10.2-Protección social (pensiones)_per_hab,2-Defensa_per_PEA,...,P.5-Formación bruta de capital_per_PEA,P.5-Formación bruta de capital_per_hab,1-Esencial por definición_per_PEA,1-Esencial por definición_per_hab,2-Esencial por adopción_per_PEA,2-Esencial por adopción_per_hab,3-Esencial Mínimo_per_PEA,3-Esencial Mínimo_per_hab,4-No esencial_per_PEA,4-No esencial_per_hab
0,1995,23377.365907,9860.236944,4253.689564,1794.145121,4092.051505,1725.968511,3501.482973,1476.875192,721.951568,...,2330.589735,983.009253,3322.156445,1401.237840,6353.257180,2679.712565,3663.776160,1545.328126,10038.176122,4233.958413
1,1996,23004.927859,9717.114985,4383.010732,1851.352003,3964.498093,1674.575294,3606.399109,1523.316875,686.879575,...,2003.008837,846.056432,3324.587173,1404.281554,6454.725296,2726.429236,3646.497193,1540.254016,9579.118198,4046.150179
2,1997,22653.347005,9631.359777,4225.047605,1796.332946,3896.880035,1656.808313,3606.972316,1533.550343,680.650359,...,2035.330306,865.346672,3448.483961,1466.166995,6391.035594,2717.230400,3378.409257,1436.373840,9435.418192,4011.588541
3,1998,23173.755892,9942.271116,4161.524448,1785.425052,3850.909406,1652.161417,3670.300437,1574.674480,670.213689,...,2185.698175,937.733354,3680.594512,1579.090963,6566.521123,2817.244369,3252.754243,1395.534013,9673.886013,4150.401771
4,1999,23473.907456,10201.214316,3906.660450,1697.743786,3853.895402,1674.813323,3784.186406,1644.519414,655.014385,...,2281.178571,991.347160,3882.552282,1687.266936,6879.868542,2989.830883,3007.835155,1307.135214,9703.651478,4216.981283


In [14]:
# verifico que no haya nulos
ratios_gastos_2023_wide.isna().sum()[ratios_gastos_2023_wide.isna().sum() > 0]

Series([], dtype: int64)

# 1.3 Calculare los gastos en funcion del PIB y en funcion de su propio peso

<br> dataframe de salida = porcent_gastos_2023_index (contiene gastos anuales expresados en %pib y %de gasto anual)

In [15]:

# Asumimos que las variables de entrada (gastos_2023_wide, datos_macro y archivos) están definidas.

# --- PASO 1: Preparación de Datos y Alineación de Índices (CORRECCIÓN DE PIB) ---

# Cargar el PIB desde el archivo macro
excel_file = pd.ExcelFile(archivos.loc[12,"ruta_archivo"])  
datos_macro = pd.read_excel(archivos.loc[12,"ruta_archivo"], sheet_name=excel_file.sheet_names[1])

# 1. Indexar y Limpiar el PIB
pib_series = (
    datos_macro[['Año', 'pib (MM € 2023)']]
    .copy()
    # 🔑 FIX: Eliminar filas donde el 'Año' esté duplicado en la fuente del PIB
    .drop_duplicates(subset=['Año'], keep='first') 
    .set_index('Año')['pib (MM € 2023)']
)

# 2. Indexar el DataFrame de Gastos (Confirmamos que está limpio)
gastos_indexado = (
    gastos_2023_wide.copy()
    .set_index('Año')
)

# 3. Obtener la serie de Gasto Total Anual (Confirmamos que está limpia)
gasto_total_anual = gastos_indexado['Gasto_Total_Anual']             

# 4. Definir las columnas de Gasto a calcular (Excluyendo Año y Gasto_Total_Anual)
# Utilizamos la lista de columnas de su DataFrame, excluyendo 'Año' (que es el índice) 
# y 'Gasto_Total_Anual' (que es el divisor).
columnas_gasto = [col for col in gastos_indexado.columns if col not in ['Gasto_Total_Anual']]

# --- PASO 2: Creación y Concatenación de Ratios ---

# 1. Lista de ratios (Inicializada para evitar la acumulación)
lista_series_ratios = [] 

# 2. Bucle de cálculo
for col in columnas_gasto: 
    serie_gasto = gastos_indexado[col]
    
    # 2.1 Gasto como % del PIB
    serie_pib = (serie_gasto.div(pib_series) * 100).rename(f'{col}_per_%pib')
    
    # 2.2 Gasto como % del Gasto Total Anual
    serie_porcent_gasto = (serie_gasto.div(gasto_total_anual) * 100).rename(f'{col}_per_%gasto_anual')
    
    # Añadimos las series calculadas
    lista_series_ratios.extend([serie_pib, serie_porcent_gasto])
    
# 3. Concatenamos todas las Series en un solo DataFrame.
porcent_gastos_2023_index = pd.concat(lista_series_ratios, axis=1)
    
# Mostrar el resultado
print(f"DataFrame de Ratios creado con {porcent_gastos_2023_index.shape[1]} nuevas series de gasto.")
print("\n--- Cabecera del DataFrame de Ratios (Indexado por Año) ---")
porcent_gastos_2023_index.head()

DataFrame de Ratios creado con 186 nuevas series de gasto.

--- Cabecera del DataFrame de Ratios (Indexado por Año) ---


,1-Servicios públicos generales_per_%pib,1-Servicios públicos generales_per_%gasto_anual,10-Protección social(sin pensiones)_per_%pib,10-Protección social(sin pensiones)_per_%gasto_anual,10.2-Protección social (pensiones)_per_%pib,10.2-Protección social (pensiones)_per_%gasto_anual,2-Defensa_per_%pib,2-Defensa_per_%gasto_anual,3-Orden público y seguridad_per_%pib,3-Orden público y seguridad_per_%gasto_anual,...,P.5-Formación bruta de capital_per_%pib,P.5-Formación bruta de capital_per_%gasto_anual,1-Esencial por definición_per_%pib,1-Esencial por definición_per_%gasto_anual,2-Esencial por adopción_per_%pib,2-Esencial por adopción_per_%gasto_anual,3-Esencial Mínimo_per_%pib,3-Esencial Mínimo_per_%gasto_anual,4-No esencial_per_%pib,4-No esencial_per_%gasto_anual
Año,,,,,,,,,,,,,,,,,,,,,
1995,8.028056,18.195761,7.722994,17.504331,6.608404,14.978090,1.362551,3.088250,1.967167,4.458626,...,4.398559,9.969428,6.269959,14.210996,11.990603,27.176959,6.914703,15.672322,18.945209,42.939723
1996,8.166610,19.052486,7.386819,17.233256,6.719594,15.676637,1.279823,2.985793,1.841325,4.295763,...,3.732090,8.706869,6.194511,14.451631,12.026715,28.058011,6.794307,15.850939,17.848216,41.639419
1997,7.737539,18.650876,7.136549,17.202226,6.605627,15.922470,1.246509,3.004635,1.832691,4.417592,...,3.727401,8.984678,6.315379,15.222845,11.704220,28.212324,6.187048,14.913510,17.279549,41.651321
1998,7.356981,17.957920,6.807858,16.617545,6.488567,15.838177,1.184842,2.892124,1.807475,4.411930,...,3.864002,9.431782,6.506766,15.882598,11.608672,28.336024,5.750405,14.036371,17.102049,41.745007
1999,6.648083,16.642566,6.558291,16.417784,6.439665,16.120820,1.114658,2.790393,1.796418,4.497087,...,3.881951,9.717933,6.607057,16.539864,11.707681,29.308578,5.118524,12.813526,16.512998,41.338032


# 1.4 Preparación de datos: INDICES MACROECONÓMICOS Y PRIVADO

<br> dataframe de salida 1 = indices_macro_wide (contiene indicadores macroeconomicos expresados en % de: ipc, deficit, deuta, tasas de empleo, indice bursatl, indice de gini,etc.)
<br> dataframe de salida 2 = valores_macro_wide (contiene info macroeconómica, salarios y demografica expresada en valor absoluto, %pib, per capita, per PEA de: PIB, deuda, deficit, etc)


In [16]:
# veo el nombre de la pestaña en forma dinamica
excel_file = pd.ExcelFile(archivos.loc[12,"ruta_archivo"])  
excel_file.sheet_names

['indices_macro',
 'valores_macro',
 'valores_demogr',
 'valores_asalariados',
 'indices_sect_privado']

In [17]:
# veo el nombre de la pestaña en forma dinamica
excel_file = pd.ExcelFile(archivos.loc[12,"ruta_archivo"])  
#importo el archivo
indices_macro_wide = pd.read_excel(archivos.loc[12,"ruta_archivo"], sheet_name=excel_file.sheet_names[0])

#indices_macro_wide.info()

indices_macro_wide.isna().sum()[indices_macro_wide.isna().sum() > 0]

Series([], dtype: int64)

In [18]:
# veo el nombre de la pestaña en forma dinamica
excel_file = pd.ExcelFile(archivos.loc[12,"ruta_archivo"])  
#importo el archivo
valores_macro = pd.read_excel(archivos.loc[12,"ruta_archivo"], sheet_name=excel_file.sheet_names[1])

#valores_macro.info()
valores_macro.isna().sum()[valores_macro.isna().sum() > 0]

Series([], dtype: int64)

In [19]:
# veo el nombre de la pestaña en forma dinamica
excel_file = pd.ExcelFile(archivos.loc[12,"ruta_archivo"])  
#importo el archivo
valores_demogr = pd.read_excel(archivos.loc[12,"ruta_archivo"], sheet_name=excel_file.sheet_names[2])
valores_demogr.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29 entries, 0 to 28
Data columns (total 8 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Año                           29 non-null     int64  
 1   total hab                     29 non-null     int64  
 2   cant ocupados                 29 non-null     float64
 3   cant desempleados             29 non-null     float64
 4   cant PEA                      29 non-null     int64  
 5   cant empleo público           29 non-null     int64  
 6   cant empleo privado           29 non-null     float64
 7   cant personas riesgo pobreza  29 non-null     int64  
dtypes: float64(3), int64(5)
memory usage: 1.9 KB


In [20]:
# veo el nombre de la pestaña en forma dinamica
excel_file = pd.ExcelFile(archivos.loc[12,"ruta_archivo"])  
#importo el archivo
valores_asalariados = pd.read_excel(archivos.loc[12,"ruta_archivo"], sheet_name=excel_file.sheet_names[3])
valores_asalariados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29 entries, 0 to 28
Data columns (total 4 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Año                        29 non-null     int64  
 1   salario bruto mediano (€)  29 non-null     float64
 2   umbral riesgo hab (€)      29 non-null     float64
 3   umbral riesgo hogar (€)    29 non-null     float64
dtypes: float64(3), int64(1)
memory usage: 1.0 KB


In [21]:
valores_macro_wide = pd.merge(valores_macro, valores_demogr, on="Año")
valores_macro_wide = valores_macro_wide.merge(valores_asalariados, on="Año")
valores_macro_wide.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29 entries, 0 to 28
Data columns (total 28 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Año                           29 non-null     int64  
 1   pib (MM € 2023)               29 non-null     int64  
 2   pib per capita                29 non-null     float64
 3   pib per PEA                   29 non-null     float64
 4   deficit (MM €)                29 non-null     float64
 5   deficit per capita            29 non-null     float64
 6   deficit per PEA               29 non-null     float64
 7   deficit_acum (MM €)           29 non-null     float64
 8   deficit_acum per capita       29 non-null     float64
 9   deficit_acum per PEA          29 non-null     float64
 10  deuda (MM €)                  29 non-null     float64
 11  deuda per capita              29 non-null     float64
 12  deuda per PEA                 29 non-null     float64
 13  balanza

# 2. TRANSFORMACION DE DATOS PARA EVALUAR SU ESTACIONAREIDAD (q no tengan tendecia)


<br>Tengo 5 dataframes:
<br> <br> IMPORTANTE: se deben eliminar los nulos (NaN) y posibles infinitos!!!!!!!!!

>    1- gastos_2023_wide (gastos COFOG expresados en MM€)
<br> 2- ratios_gastos_2023_wide (gastos COFOG expresados en € per capita y # per PAE)
<br> 3- valores_macro_wide (contiene valores macroeconomicos, salarios y demograficos)

>    4- porcent_gastos_2023_index (gastos COFOG expresados en %pib y %gasto anual)
<br> 5- indices_macro_wide (contiene indicadores, tasas, indices,etc)



<br> El objetivo es convertir las series no estacionarias (que tienen tendencia) en series estacionarias (sin tendencia) para luego aplicar el test de ADF para comprobarlo

<br> El output sera una mega tabla llamada "df_master" que contendra los datos "transformados".

# 2.1 Transformacion de datos "absolutos" en variacion porcentual anual (% = pct)

<br> Aplico esta trasnformacion a:
>    1- gastos_2023_wide (gastos COFOG expresados en MM€) -----------------------------------> df_transf_gastos
<br> 2- ratios_gastos_2023_wide (gastos COFOG expresados en € per capita y # per PAE) ---------> df_transf_ratios_gastos
<br> 3- valores_macro_wide (contiene valores macroeconomicos, salarios y demograficos) ---------> df_transf_macro

In [22]:
# ----------------------------------------------------
# 1. Transformación de datos de VALORES ABSOLUTAS (variación %)
# ----------------------------------------------------

#### 1- gastos_2023_wide (gastos COFOG expresados en MM€) ####
# reasigno para no estropear el original
df_transf_gastos = gastos_2023_wide

for col in df_transf_gastos.columns[1:]:
    # Creación de la columna con sufijo '_Crecimiento'
    df_transf_gastos[f'{col}_var'] = df_transf_gastos[col].pct_change() * 100
     # Eliminamos la columna original para no confundir
    df_transf_gastos = df_transf_gastos.drop(columns=[col])
    df_transf_gastos = df_transf_gastos.fillna(0)
    df_transf_gastos.replace([np.inf, -np.inf], 0, inplace=True)

#  Eliminar la primera fila (todas las pct_change tienen NaN ahí)
df_transf_gastos = df_transf_gastos.iloc[1:]


#### 2- ratios_gastos_2023_wide (gastos COFOG expresados en € per capita y # per PAE) ####
# reasigno para no estropear el original
df_transf_ratios_gastos = ratios_gastos_2023_wide

for col in df_transf_ratios_gastos.columns[1:]:
    # Creación de la columna con sufijo '_Crecimiento'
    df_transf_ratios_gastos[f'{col}_var'] = df_transf_ratios_gastos[col].pct_change() * 100
     # Eliminamos la columna original para no confundir
    df_transf_ratios_gastos = df_transf_ratios_gastos.drop(columns=[col])
    df_transf_ratios_gastos = df_transf_ratios_gastos.fillna(0)
    df_transf_ratios_gastos.replace([np.inf, -np.inf], 0, inplace=True)

#  Eliminar la primera fila (todas las pct_change tienen NaN ahí)
df_transf_ratios_gastos = df_transf_ratios_gastos.iloc[1:]


#### 3- valores_macro_wide (contiene valores macroeconomicos, salarios y demograficos) ####
# reasigno para no estropear el original
df_transf_macro = valores_macro_wide
                                        
for col in df_transf_macro.columns[1:]:
    # Creación de la columna con sufijo '_Crecimiento'
    df_transf_macro[f'{col}_var'] = df_transf_macro[col].pct_change() * 100
    # Eliminamos la columna original para no confundir
    df_transf_macro = df_transf_macro.drop(columns=[col])
    df_transf_macro = df_transf_macro.fillna(0)
    df_transf_macro.replace([np.inf, -np.inf], 0, inplace=True)

#  Eliminar la primera fila (todas las pct_change tienen NaN ahí)
df_transf_macro = df_transf_macro.iloc[1:]

#### SE VERIFICA QUE NO HAY NULOS
df_transf_ratios_gastos.isna().sum()[df_transf_ratios_gastos.isna().sum() > 0]


Series([], dtype: int64)

# 2.2 Transformacion de datos "relativo o tasas" en variacion anuales (delta = diff)

<br> Aplico esta transformación a:
>    4- porcent_gastos_2023_index (gastos COFOG expresados en %pib y %gasto anual).  -----> df_transf_porcentaje_gastos
<br> 5- indices_macro_wide (contiene indicadores, tasas, indices,etc). --------------------------> df_transf_indices

In [23]:
# ----------------------------------------------------
# 2. Transformación de datos de RATIOS/PORCENTAJES (Cambio Absoluto)
# ----------------------------------------------------

#### 4- porcent_gastos_2023_index (gastos COFOG expresados en %pib y %gasto anual)
df_transf_porcentaje_gastos = porcent_gastos_2023_index.copy()

for col in df_transf_porcentaje_gastos.columns[1:]:
    # Creación de la columna con sufijo '_Cambio'
    df_transf_porcentaje_gastos[f'{col}_dif'] = df_transf_porcentaje_gastos[col].diff()
    # Eliminamos la columna original
    df_transf_porcentaje_gastos = df_transf_porcentaje_gastos.drop(columns=[col])
    df_transf_porcentaje_gastos = df_transf_porcentaje_gastos.fillna(0)

# verifico que no haya nulos
df_transf_porcentaje_gastos.isna().sum()[df_transf_porcentaje_gastos.isna().sum() > 0]

#### 5- indices_macro_wide (contiene indicadores, tasas, indices,etc)
df_transf_indices = indices_macro_wide

for col in df_transf_indices.columns[1:]:
    # Creación de la columna con sufijo '_Cambio'
    df_transf_indices[f'{col}_dif'] = df_transf_indices[col].diff()
    # Eliminamos la columna original
    df_transf_indices = df_transf_indices.drop(columns=[col])
    df_transf_indices = df_transf_indices.fillna(0)

# verifico que no haya nulos
df_transf_indices.isna().sum()[df_transf_indices.isna().sum() > 0]


/var/folders/18/mfrphkvn4wq2mf7pmp8_5m2c0000gn/T/ipykernel_79869/2493187608.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_transf_porcentaje_gastos[f'{col}_dif'] = df_transf_porcentaje_gastos[col].diff()
/var/folders/18/mfrphkvn4wq2mf7pmp8_5m2c0000gn/T/ipykernel_79869/2493187608.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_transf_porcentaje_gastos[f'{col}_dif'] = df_transf_porcentaje_gastos[col].diff()
/var/folders/18/mfrphkvn4wq2mf7pmp8_5m2c0000gn/T/ipykernel_79869/2493187608.py:10: PerformanceWarning: DataF

Series([], dtype: int64)

# 2.3 Armo el master dataframe

<br> datafram de la salida = df_master_transf (contiene todos los datos transformados, sin nulos, sin infinitos) (elimina 1 fila)

In [24]:
# hago un merge por orden de los dataframes transformados
df_master_transf = pd.merge(df_transf_gastos, df_transf_ratios_gastos, on="Año")
df_master_transf = df_master_transf.merge(df_transf_macro, on="Año")
df_master_transf = df_master_transf.merge(df_transf_porcentaje_gastos, on="Año")
df_master_transf = df_master_transf.merge(df_transf_indices, on="Año")

# una vez hecho el merge, elimino el año
#df_master_transf = df_master_transf.drop("Año", axis=1)
df_master_transf = df_master_transf.set_index('Año')

# eliminamos primer fila (año 1995) q tendrá NaN después de .pct_change() o .diff()
df_master_transf = df_master_transf.dropna()
df_master_transf = df_master_transf.iloc[1:]

# verifico nulos
df_master_transf.isna().sum()[df_master_transf.isna().sum() > 0]

# verifico infinitos
df_master_transf[df_master_transf.isin([np.inf, -np.inf]).any(axis=1)]

df_master_transf.shape

(27, 505)

# 3. APLICACION DEL TEST DE DICKEY-FULLER AUMENTADO (ADF)

<br> El objetivo es definir que campos, variables, indices, etc se comportan en forma estacionaria y por lo tanto "serían" plausibles de aplicar la correlación

# 3.1 Aplicamos el Test ADF

<br> Se define como límite un p-value <= a 0.05.
<br> Todo lo que tiene mayor valor se descarta en esta primera iteración

<br> dataframes de salida:
>    df_resumen_adf (contiene resultados del test, pares, p-value, )

>    df_estacionario_1 (series estacionarias que cumpler el test)
<br> df_a_trasnformar_2nda (series NO estacionarias (a doble diferenciar))

In [25]:
# ⚠️ Asegúrate de que tu df_master_transf es el DataFrame resultante del Paso 2 
# (con todas las series ya diferenciadas por primera vez y con los infinitos reemplazados).

def run_adf_test(serie, umbral=0.05):
    """
    Ejecuta el Test ADF de forma robusta y devuelve el p-value y el estado.
    """
    # 1. Limpieza (manejo de posibles NaN restantes y asegurar longitud mínima)
    serie_limpia = serie.dropna().replace([np.inf, -np.inf], np.nan).dropna()
    
    if len(serie_limpia) < 8:
        return {'p_value': 1.0, 'status': '❌ DATOS INSUFICIENTES', 'stationary': False}

    try:
        resultado = adfuller(serie_limpia.values)  
        p_value = resultado[1]
    except ValueError:
        # Esto ocurre si todos los datos restantes son idénticos (varianza cero)
        return {'p_value': 1.0, 'status': '❌ VARIANZA CERO', 'stationary': False}
    
    # Criterio Estadístico: Rechazar H0 si p-value < umbral (0.05)
    if p_value < umbral:
        return {'p_value': p_value, 'status': '✅ ESTACIONARIA', 'stationary': True}
    else:
        return {'p_value': p_value, 'status': '❌ NO ESTACIONARIA', 'stationary': False}

# -------------------------------------------------------------------
# A. ITERACIÓN Y RECOLECCIÓN DE RESULTADOS
# -------------------------------------------------------------------
resultados_adf_1 = []

for columna in df_master_transf.columns:                # como el "Año" es indice, el for no lo itera
    resultado = run_adf_test(df_master_transf[columna])
    
    resultados_adf_1.append({
        'Serie': columna,
        'p_value': resultado['p_value'],
        'Estado': resultado['status'],
        'Estacionaria': resultado['stationary']
    })

# Convertir a DataFrame para el resumen
df_resumen_adf = pd.DataFrame(resultados_adf_1)

# -------------------------------------------------------------------
# B. FILTRADO Y CREACIÓN DE LOS NUEVOS DATAFRAMES
# -------------------------------------------------------------------

# 1. Series Estacionarias (listas para el Paso 3)
columnas_estacionarias = df_resumen_adf[df_resumen_adf['Estacionaria'] == True]['Serie'].tolist()
df_estacionario_1 = df_master_transf[columnas_estacionarias].copy()

# 2. Series No Estacionarias (a doble diferenciar)
columnas_a_doble_diferenciar = df_resumen_adf[df_resumen_adf['Estacionaria'] == False]['Serie'].tolist()
df_a_trasnformar_2nda = df_master_transf[columnas_a_doble_diferenciar].copy()

print("--- RESULTADOS RESUMEN DEL TEST ADF ---")
print(f"Total de series: {len(df_master_transf.columns[1:])}")
print(f"Series Estacionarias (✅): {len(columnas_estacionarias)}")
print(f"Series a doble diferenciar (❌): {len(columnas_a_doble_diferenciar)}")
print("\nTABLA RESUMEN DE LAS SERIES NO ESTACIONARIAS:")

# Mostrar solo las series problemáticas para fácil revisión
df_resumen_adf[df_resumen_adf['Estacionaria'] == False].sort_values(by='p_value', ascending=False)


--- RESULTADOS RESUMEN DEL TEST ADF ---
Total de series: 504
Series Estacionarias (✅): 366
Series a doble diferenciar (❌): 139

TABLA RESUMEN DE LAS SERIES NO ESTACIONARIAS:


,Serie,p_value,Estado,Estacionaria
26,10.8-I+D relacionada con la protección social_var,1.000000,❌ NO ESTACIONARIA,False
147,10.8-I+D relacionada con la protección social_per_hab_var,1.000000,❌ NO ESTACIONARIA,False
146,10.8-I+D relacionada con la protección social_per_PEA_var,1.000000,❌ NO ESTACIONARIA,False
128,1.6-Servicios públicos generales n.c.o.p._per_PEA_var,0.979542,❌ NO ESTACIONARIA,False
129,1.6-Servicios públicos generales n.c.o.p._per_hab_var,0.979240,❌ NO ESTACIONARIA,False
...,...,...,...,...
123,1.3-Servicios generales_per_hab_var,0.056464,❌ NO ESTACIONARIA,False
4,2-Defensa_var,0.056124,❌ NO ESTACIONARIA,False
115,8-Ocio. cultura y religión_per_hab_var,0.053055,❌ NO ESTACIONARIA,False
125,1.4-Investigación básica_per_hab_var,0.052526,❌ NO ESTACIONARIA,False


# 3.2 Filtramos y seleccionamos datos para aplicar Correlación Cruzada (CCF)

Se excluiran los datos de los campos que no cumplan con el cirterio tras la 1era transformaciones. Se mantiene 27 registros de los 30 originales.

<br> dataframe de salida = df_analisis_1_transf (contiene todos los campos estacionarios, y Año como indice)

In [26]:
# ⚠️ Asegúrate de que tienes df_estacionario_1 y df_doble_diferenciado.
df_analisis_1_transf = df_estacionario_1.copy()

N_final = df_analisis_1_transf.shape[0]
C_final = df_analisis_1_transf.shape[1]

print("\n--- DATAFRAME FINAL CONSOLIDADO Y ESTACIONARIO (tras 1era transformación) ---")
print(f"Período de Análisis Final (N): {N_final} años")
print(f"Total de Series Estacionarias (C): {C_final}")
print(f"Series Excluidas (no estacionarias): {len(columnas_a_doble_diferenciar)}")


--- DATAFRAME FINAL CONSOLIDADO Y ESTACIONARIO (tras 1era transformación) ---
Período de Análisis Final (N): 27 años
Total de Series Estacionarias (C): 366
Series Excluidas (no estacionarias): 139


# 4 NORMALIZACIÓN DE Z-SCORE

<br> Una vez confirmada la estacionariedad, aplicar el Z-score a las series que vas a comparar con CCF. Esto es esencial cuando deseas comparar series que tienen unidades y magnitudes totalmente diferentes.

<br> El Z-score es una transformación lineal que simplemente cambia la escala y el centro de tus datos.
<br> formula: Z = (valor - media) / desv estadar
<br> Lo que hace: Traslada la media ($\mu$) a cero y la Desviación Estándar ($\sigma$) a uno.
<br> Lo que NO hace: El Z-score preserva la forma y las dependencias temporales de la serie original. Si tu serie original tiene una tendencia creciente (no estacionaria), la serie Z-score simplemente tendrá esa misma tendencia creciente, pero con su media centrada en cero.

# 4.1 Aplicamos la normalizacion al dataframe filtrado luego del Test ADF

<br> datafram de salida = df_analisis_1_transf_norm_z (contiene los datos transformado ahora expresados en z-scores).

<br> Un Z-score se interpreta como el número de desviaciones estándar que un valor se aleja de su media (cero).
>  Valor Cero (ej. -0.005095 en deficit (% pib)_dif en 2000) -> Interpretación: El crecimiento de ese año fue exactamente igual al promedio histórico de crecimiento de la serie -> ImplicacionEconomica: Fue un año "normal" para esa variable.

> Positivo (ej. 1.500425 en índice IBEX35_dif en 1998) -> Interpretación: El crecimiento de ese año fue superior a la media histórica en 1.50 desviaciones estándar. -> ImplicacionEconomica: Fue un año de fuerte crecimiento (o alta volatilidad positiva) para el IBEX 35.

> Negativo (ej. -2.216002 en interes bonos 10 años (% anual)_dif en 1997) -> Interpretación: El crecimiento de ese año fue inferior a la media histórica en 2.22 desviaciones estándar.-> ImplicacionEconomica: Fue un evento extremo (o de muy baja volatilidad) para la variación del interés de los bonos.

In [27]:
# ----------------------------------------------------
# 1. Aplicar Normalización Z-SCORE a todo el DataFrame
# ----------------------------------------------------
df_analisis_1_transf_norm_z = (
    df_analisis_1_transf - df_analisis_1_transf.mean()
) / df_analisis_1_transf.std()

# La fórmula del Z-score se aplica vectorialmente a cada columna:
# Z = (Valor - Media) / Desviación Estándar
# Pandas calcula automáticamente la media y la desviación estándar para CADA columna.

# ----------------------------------------------------
# 2. Verificación de la Transformación (Script Corregido)
# ----------------------------------------------------

print("--- DataFrame Normalizado (Z-score) ---")
print(f"Número de series (columnas) normalizadas: {len(df_analisis_1_transf_norm_z.columns)}")

# Sustitución de .to_markdown() por la impresión estándar de Pandas
df_analisis_1_transf_norm_z.head()

print("\n--- Estadísticas de Verificación del Z-score ---")
print("Media de cada columna (debe ser cercana a 0.00):")
df_analisis_1_transf_norm_z.mean().round(2)
print("\nDesviación Estándar de cada columna (debe ser 1.00):")
df_analisis_1_transf_norm_z.std().round(2)

--- DataFrame Normalizado (Z-score) ---
Número de series (columnas) normalizadas: 366

--- Estadísticas de Verificación del Z-score ---
Media de cada columna (debe ser cercana a 0.00):

Desviación Estándar de cada columna (debe ser 1.00):


Gasto_Total_Anual_var                      1.0
1-Servicios públicos generales_var         1.0
10-Protección social(sin pensiones)_var    1.0
10.2-Protección social (pensiones)_var     1.0
3-Orden público y seguridad_var            1.0
                                          ... 
deficit (% pib)_dif                        1.0
balanza com (% pib)_dif                    1.0
tasa riesgo pobreza (% pobl)_dif           1.0
tasa ocupación (% PEA)_dif                 1.0
interes bonos 10 años (% anual)_dif        1.0
Length: 366, dtype: float64

# 4.2 LIstado de variables derivadas que se comportan idual (misma señal)

<br> Se aplica correlación simple para ver si hay duplicidad o ruido (Colinealidad multiple)

In [28]:
# ⚠️ ASUME que tu DataFrame de entrada es el ESTACIONARIO y Z-SCORE NORMALIZADO
df_input = df_analisis_1_transf_norm_z 

# 1. Definición del Umbral de Redundancia Extrema
# Usaremos 0.999 para aislar solo los casos más problemáticos (ej. var vs. per_hab)
umbral_redundancia = 0.95 

# 2. Cálculo y Filtrado de la Matriz de Correlación
matriz_corr_abs = df_input.corr().abs()

# Convertir la matriz en una serie de pares, eliminando la diagonal (r=1) y duplicados
df_redundancia = matriz_corr_abs.unstack()
df_redundancia = df_redundancia[df_redundancia.index.get_level_values(0) < df_redundancia.index.get_level_values(1)]
df_redundancia = df_redundancia.sort_values(ascending=False)

# Filtrar por el umbral de redundancia extrema
pares_redundantes = df_redundancia[df_redundancia >= umbral_redundancia]

print("\n--- DIAGNÓSTICO: PARES CON REDUNDANCIA EXTREMA (r >= +/-0.999) ---")

if not pares_redundantes.empty:
    print(f"Total de pares redundantes identificados: {len(pares_redundantes)}")
    print("-----> Estos pares son esencialmente la misma señal.")
    # Muestra los primeros 30 pares para una revisión completa de la redundancia
    print("\n",pares_redundantes.to_string())
else:
    print(f"✅ No se encontraron pares con correlación simultánea superior a +/-{umbral_redundancia}.")


--- DIAGNÓSTICO: PARES CON REDUNDANCIA EXTREMA (r >= +/-0.999) ---
Total de pares redundantes identificados: 293
-----> Estos pares son esencialmente la misma señal.

 10.2-Edad avanzada(pensiones)_var                                                                      10.2-Protección social (pensiones)_var                                                                   1.000000
10.2-Edad avanzada(pensiones)_per_%pib_dif                                                             10.2-Protección social (pensiones)_per_%pib_dif                                                          1.000000
10.2-Edad avanzada(pensiones)_per_hab_var                                                              10.2-Protección social (pensiones)_per_hab_var                                                           1.000000
10.2-Edad avanzada(pensiones)_per_%gasto_anual_dif                                                     10.2-Protección social (pensiones)_per_%gasto_anual_dif                      

# 4.3 Elimino los campos con alta colinealidad (k=0)

In [29]:

# ⚠️ ASUME que tu DataFrame de entrada es el ESTACIONARIO y Z-SCORE NORMALIZADO
df_input = df_analisis_1_transf_norm_z 
umbral_colinealidad = 0.95 # Umbral de eliminación estricto para series derivadas
columnas_a_eliminar = set()
matriz_corr_abs = df_input.corr().abs()

# 1. PASO: ITERACIÓN Y ELIMINACIÓN DE REDUNDANCIAS EXTREMAS (r >= 0.999)
for col1 in matriz_corr_abs.columns:
    for col2 in matriz_corr_abs.columns:
        
        if col1 == col2 or col1 in columnas_a_eliminar or col2 in columnas_a_eliminar:
            continue
            
        if matriz_corr_abs.loc[col1, col2] >= umbral_colinealidad:
            
            # --- Lógica de Priorización para Eliminar la Versión Derivada ---
            
            # Identificar si la columna es una derivación (per_hab, per_PEA, per_%gasto_anual, etc.)
            es_derivada_1 = any(sub in col1 for sub in ['_per_hab', '_per_PEA', '_per_%gasto_anual', '_per_%pib'])
            es_derivada_2 = any(sub in col2 for sub in ['_per_hab', '_per_PEA', '_per_%gasto_anual', '_per_%pib'])
            
            # Si ambas son derivadas o ambas son base (ej. duplicados por coma/punto)
            if es_derivada_1 == es_derivada_2:
                # Caso A: Duplicado (r=1.0) o casi perfecto en el mismo nivel: Eliminar la de nombre más largo/menos estándar
                if len(col1) >= len(col2):
                    columnas_a_eliminar.add(col1)
                else:
                    columnas_a_eliminar.add(col2)
            
            # Caso B: Una es derivada y la otra es base. CONSERVAR la versión base o más simple.
            elif es_derivada_1 and not es_derivada_2:
                columnas_a_eliminar.add(col1) # Eliminar la derivada (col1)
            elif es_derivada_2 and not es_derivada_1:
                columnas_a_eliminar.add(col2) # Eliminar la derivada (col2)

# 2. Creación del nuevo DataFrame limpio
columnas_limpias = list(set(df_input.columns) - columnas_a_eliminar)
df_analisis_final_limpio = df_input[columnas_limpias].copy()

# 3. Resumen y Preparación para el siguiente paso
print(f"\n--- RESULTADO DE LA LIMPIEZA DE REDUNDANCIA ---")
print(f"Variables originales: {len(df_input.columns)}")
print(f"Variables ELIMINADAS (por redundancia r >= 0.999): {len(columnas_a_eliminar)}")
print(f"Variables FINALES que pasan a Granger: {len(df_analisis_final_limpio.columns)}")

# ⚠️ PASO CRÍTICO: FILTRAR LOS CANDIDATOS CCF
# La lista de candidatos CCF debe alinearse con estas nuevas columnas.
# El siguiente paso debe ser el **doble filtro** del script anterior, usando:
# 1. df_analisis_final_limpio (para el filtro de existencia)
# 2. df_ccf_exhaustivo (para el filtro de coeficiente).

print("\n✅ DataFrame 'df_analisis_final_limpio' listo para ser usado con el Test de Granger.")


--- RESULTADO DE LA LIMPIEZA DE REDUNDANCIA ---
Variables originales: 366
Variables ELIMINADAS (por redundancia r >= 0.999): 166
Variables FINALES que pasan a Granger: 200

✅ DataFrame 'df_analisis_final_limpio' listo para ser usado con el Test de Granger.


# 5. CORRELACION CRUZADA (CCF)

Objetivo, es identificar, para cada uno de los pares posibles del dataframe, , el número óptimo de años de desfase ($k$) que maximiza la correlación entre la causa rezagada y el efecto actual.

# 5.1 Aplico el método de correlación cruzada con desfasage (CCF)

<br> dataframe de entrada = df_analisis_final_limpio (datos donde hemos quitado las señales con multiple colinealidad (variables derivadas similares desde el punto de vist etaditico)

<br> dataframe de salida = resultados_ccf (contiene todas las combinaciones posibles de variables, esl desfasage y el p_value)

In [30]:
### INPUT ###
#df_input_ccf = df_analisis_1_transf_norm_z                # tiene todoas las variables derivadsa y madre
df_input_ccf = df_analisis_final_limpio                 # sin variables con multiple colinealidad

# 1. Definición de todas las series
columnas_totales = df_input_ccf.columns.tolist()
num_series = len(columnas_totales)

print(f"Series Totales a analizar: {num_series}")

# 2. Parámetros y Colección de Resultados
max_lags_a_revisar = 7
umbral_minimo_coef = 0.3
resultados_lista = [] 

# El número total de pares a analizar es (N * (N-1) * K_max)
print(f"\n--- BÚSQUEDA DEL REZAGO (K) ÓPTIMO ({num_series} series, K máx={max_lags_a_revisar}) ---")


# 3. Iteración Exhaustiva: Cada Serie (Y) es CAUSA de Cada Serie (X)
# El resultado será una tabla de N * (N-1) posibles relaciones
print("\nCorriendo CCF: TODAS las combinaciones posibles...")

for i in range(num_series):
    for j in range(num_series):
        
        nombre_causa = columnas_totales[i]
        nombre_efecto = columnas_totales[j]
        
        # Evitar comparar una serie consigo misma
        if nombre_causa == nombre_efecto:
            continue
            
        # X es el EFECTO (variable actual), Y es la CAUSA (variable rezagada)
        serie_causa = df_input_ccf[nombre_causa].dropna()
        serie_efecto = df_input_ccf[nombre_efecto].dropna()
        
        min_len = min(len(serie_causa), len(serie_efecto))
        
        if min_len < max_lags_a_revisar + 1:
            continue # Datos insuficientes

        # ccf(x=EFECTO, y=CAUSA)
        ccf_valores = ccf(x=serie_efecto[:min_len], y=serie_causa[:min_len], adjusted=False)
        
        # Solo miramos rezagos positivos k=1, 2, 3... (Causa precede Efecto)
        lags_positivos = ccf_valores[1 : max_lags_a_revisar + 1] 
        
        if len(lags_positivos) > 0:
            k_idx = np.argmax(np.abs(lags_positivos))
            k_optimo = k_idx + 1
            coef_max = lags_positivos[k_idx]
            
            if np.abs(coef_max) >= umbral_minimo_coef:
                 resultados_lista.append({
                    'CAUSA': nombre_causa, 
                    'EFECTO': nombre_efecto, 
                    'k_optimo': k_optimo, 
                    'Coeficiente': coef_max,
                })

# 4. CONSTRUCCIÓN Y FORMATO DEL DATAFRAME FINAL
resultados_ccf = pd.DataFrame(resultados_lista)

print(f"\n✅ DataFrame 'df_ccf_exhaustivo' creado con {len(resultados_ccf)} pare de candidatos.")


Series Totales a analizar: 200

--- BÚSQUEDA DEL REZAGO (K) ÓPTIMO (200 series, K máx=7) ---

Corriendo CCF: TODAS las combinaciones posibles...

✅ DataFrame 'df_ccf_exhaustivo' creado con 18194 pare de candidatos.


# 5.2 Listado de correlaciones "ALTAS"

In [31]:
# Criterio de aceptacion simétrico
umbral_min = 0.8
umbral_max = .95
criterio_granger = ((resultados_ccf["Coeficiente"]>=umbral_min) & (resultados_ccf["Coeficiente"]<umbral_max))| ((resultados_ccf["Coeficiente"]<=(-umbral_min)) & (resultados_ccf["Coeficiente"]>(-umbral_max)))
df_candidatos_granger = resultados_ccf[criterio_granger].copy()

print(f"\n--- FILTRO: CANDIDATOS CON CORRELACIÓN ENTRE -> {umbral_min}>= |r| <{umbral_max} ---")
print(f"Total de pares con una correlación ALTA: {len(df_candidatos_granger)}")

# Muestra los candidatos de la lista filtrada
print(f"\n--- CANDIDATOS ORDENADOS POR CAUSA CON CORRELACIÓN ENTRE -> {umbral_min}>= |r| <{umbral_max} ---")
df_candidatos_granger



--- FILTRO: CANDIDATOS CON CORRELACIÓN ENTRE -> 0.8>= |r| <0.95 ---
Total de pares con una correlación ALTA: 6

--- CANDIDATOS ORDENADOS POR CAUSA CON CORRELACIÓN ENTRE -> 0.8>= |r| <0.95 ---


,CAUSA,EFECTO,k_optimo,Coeficiente
2075,9.5-Educación no atribuible a ningún nivel_per_%gasto_anual_dif,5.1-Gestión de los residuos_per_%pib_dif,1,0.812792
6299,9.5-Educación no atribuible a ningún nivel_var,5.1-Gestión de los residuos_per_%pib_dif,1,0.818142
6305,9.5-Educación no atribuible a ningún nivel_var,1.3-Servicios generales_var,1,0.806251
6310,9.5-Educación no atribuible a ningún nivel_var,5.1-Gestión de los residuos_var,1,0.851994
10081,9.3-Educación postsecundaria no terciaria_var,7.2-Servicios ambulatorios_per_hab_var,1,-0.926587
10145,9.3-Educación postsecundaria no terciaria_var,7.3-Servicios hospitalarios_per_%pib_dif,1,0.880680


# 6. TEST DE CAUSALIDAD DE GRANGER

# 6.1 Filtro pares de correlacionadas para evaluar Causa-Efecto con test de Granger

<br> dataframe de salida = df_candidatos_granger (contiene las combinaciones posibles de variables causa-efecto filtradas por algun criterio)

In [45]:
# Criterio de aceptacion simétrico
umbral_min = 0.80
umbral_max = 0.95
criterio_granger = ((resultados_ccf["Coeficiente"]>=umbral_min) & (resultados_ccf["Coeficiente"]<=umbral_max))| ((resultados_ccf["Coeficiente"]<=(-umbral_min)) & (resultados_ccf["Coeficiente"]>=(-umbral_max)))
df_candidatos_granger = resultados_ccf[criterio_granger].copy()

print(f"\n--- FILTRO: CANDIDATOS CON CORRELACIÓN ENTRE -> {umbral_min}>= |r| <={umbral_max} ---")
print(f"Total 5 de candidatos filtrados para testear con Granger: {len(df_candidatos_granger)}")

# Muestra los candidatos de la lista filtrada
df_candidatos_granger.head()


--- FILTRO: CANDIDATOS CON CORRELACIÓN ENTRE -> 0.8>= |r| <=0.95 ---
Total 5 de candidatos filtrados para testear con Granger: 6


,CAUSA,EFECTO,k_optimo,Coeficiente
2075,9.5-Educación no atribuible a ningún nivel_per_%gasto_anual_dif,5.1-Gestión de los residuos_per_%pib_dif,1,0.812792
6299,9.5-Educación no atribuible a ningún nivel_var,5.1-Gestión de los residuos_per_%pib_dif,1,0.818142
6305,9.5-Educación no atribuible a ningún nivel_var,1.3-Servicios generales_var,1,0.806251
6310,9.5-Educación no atribuible a ningún nivel_var,5.1-Gestión de los residuos_var,1,0.851994
10081,9.3-Educación postsecundaria no terciaria_var,7.2-Servicios ambulatorios_per_hab_var,1,-0.926587


# 6.2 Aplico el Test de Granger

<br> dataframe de salida = df_causalidad_detectada (contiene pares de vacriables que estadisticamente tienen un comportamiento causa-efecto)

In [33]:
# Y que el DataFrame de datos Z-score se llama df_analisis_1_transf_norm_z
resultados_ccf = resultados_ccf

# defino usar el df limpio de multpler colinealidad
# df_datos = df_analisis_1_transf_norm_z

df_datos = df_analisis_final_limpio

# 1. Definición del Criterio de Aceptación Simétrico
print(f"\n--- FILTRO: CANDIDATOS CON CORRELACIÓN ENTRE -> {umbral_min}>= |r| <={umbral_max} ---")
print(f"Total de candidatos filtrados: {len(df_candidatos_granger)}")
print(df_candidatos_granger.head().to_string())

# 2. Ejecución del Test de Causalidad de Granger

resultados_granger_final = []

print(f"\n--- INICIANDO TEST DE CAUSALIDAD DE GRANGER (Total de pruebas: {len(df_candidatos_granger)}) ---")

# Iterar sobre los candidatos CCF filtrados
for index, row in df_candidatos_granger.iterrows():
    
    causa = row['CAUSA']
    efecto = row['EFECTO']
    lags_granger = int(row['k_optimo']) # Usar el k_optimo encontrado en CCF
    
    # El test de Granger necesita que la CAUSA (Y) y el EFECTO (X) estén en un solo DataFrame
    # con el EFECTO primero (X) y la CAUSA en segundo lugar (Y) para la prueba X ~ Y.
    data_test = df_datos[[efecto, causa]].dropna()
    
    # ⚠️ grangercausalitytests(df, max_lag, verbose=False)
    # H0 (Hipótesis Nula): CAUSA NO Granger-causa EFECTO.
    
    try:
        granger_test_output = grangercausalitytests(
            data_test, 
            max_lag=[lags_granger], 
            verbose=False # Suprime la salida detallada por cada test
        )
        
        # Extraer el valor P del test F (el más común y robusto) para el lag óptimo
        # La estructura es: {lag: [estadísticas], ...} -> buscar el test F (index 0 de los tests)
        p_valor_f = granger_test_output[lags_granger][0]['ssr_ftest'][1]
        
        # Determinar si se rechaza H0 (si p < 0.05, sí hay Causalidad)
        rechazo_h0 = p_valor_f < 0.05
        
        resultados_granger_final.append({
            'CAUSA_Y': causa,
            'EFECTO_X': efecto,
            'k_optimo': lags_granger,
            'CCF_Coeficiente': row['Coeficiente'],
            'Granger_P_valor': p_valor_f,
            'Causalidad_Significativa (p<0.05)': rechazo_h0
        })
        
    except Exception as e:
        # Manejar errores (ej. DataError por series muy cortas)
        resultados_granger_final.append({
            'CAUSA_Y': causa,
            'EFECTO_X': efecto,
            'k_optimo': lags_granger,
            'CCF_Coeficiente': row['Coeficiente'],
            'Granger_P_valor': f"ERROR: {e}",
            'Causalidad_Significativa (p<0.05)': False
        })

# 3. Construcción y Formato del DataFrame Final de Granger
df_granger_resultados = pd.DataFrame(resultados_granger_final)

# Filtrar solo las relaciones que son estadísticamente significativas (Causalidad)
df_causalidad_detectada = df_granger_resultados[
    df_granger_resultados['Causalidad_Significativa (p<0.05)'] == True
].sort_values(
    by='Granger_P_valor', 
    ascending=True
).reset_index(drop=True)

print(f"\n--- RESULTADOS FINALES DE CAUSALIDAD DE GRANGER ---")
print(f"Total de relaciones causalidad significativas (p < 0.05): {len(df_causalidad_detectada)}")

# Mostrar el Top 10 de las causalidades más fuertes (p-valor más bajo)
if not df_causalidad_detectada.empty:
    print(df_causalidad_detectada.head(10).to_string())
else:
    print("\nNo se detectó ninguna relación de causalidad estadísticamente significativa (p < 0.05) entre los candidatos CCF filtrados.")


--- FILTRO: CANDIDATOS CON CORRELACIÓN ENTRE -> 0.85>= |r| <=0.95 ---
Total de candidatos filtrados: 3
                                                CAUSA                                    EFECTO  k_optimo  Coeficiente
6310   9.5-Educación no atribuible a ningún nivel_var           5.1-Gestión de los residuos_var         1     0.851994
10081   9.3-Educación postsecundaria no terciaria_var    7.2-Servicios ambulatorios_per_hab_var         1    -0.926587
10145   9.3-Educación postsecundaria no terciaria_var  7.3-Servicios hospitalarios_per_%pib_dif         1     0.880680

--- INICIANDO TEST DE CAUSALIDAD DE GRANGER (Total de pruebas: 3) ---

--- RESULTADOS FINALES DE CAUSALIDAD DE GRANGER ---
Total de relaciones causalidad significativas (p < 0.05): 0

No se detectó ninguna relación de causalidad estadísticamente significativa (p < 0.05) entre los candidatos CCF filtrados.


# 8. Evaluo las correlaciones encontradas:

In [46]:
df_candidatos_granger

,CAUSA,EFECTO,k_optimo,Coeficiente
2075,9.5-Educación no atribuible a ningún nivel_per_%gasto_anual_dif,5.1-Gestión de los residuos_per_%pib_dif,1,0.812792
6299,9.5-Educación no atribuible a ningún nivel_var,5.1-Gestión de los residuos_per_%pib_dif,1,0.818142
6305,9.5-Educación no atribuible a ningún nivel_var,1.3-Servicios generales_var,1,0.806251
6310,9.5-Educación no atribuible a ningún nivel_var,5.1-Gestión de los residuos_var,1,0.851994
10081,9.3-Educación postsecundaria no terciaria_var,7.2-Servicios ambulatorios_per_hab_var,1,-0.926587
10145,9.3-Educación postsecundaria no terciaria_var,7.3-Servicios hospitalarios_per_%pib_dif,1,0.880680


## Causa-Efecto 14859

In [36]:
causa_14859_var = pd.DataFrame(df_analisis_final_limpio[["9.5-Educación no atribuible a ningún nivel_var", "5.1-Gestión de los residuos_var"]]).reset_index()
causa_14859_var.head()

,Año,9.5-Educación no atribuible a ningún nivel_var,5.1-Gestión de los residuos_var
0,1997,-0.240545,0.111031
1,1998,0.649637,0.096701
2,1999,-2.827363,0.038772
3,2000,3.446404,-2.473674
4,2001,-0.055044,3.893019


In [37]:
df = causa_14859_var

fig = go.Figure()
# Barras: IPC anual (eje izquierdo)
fig.add_trace(go.Bar(
    x=df["Año"],
    y=df[df.columns[1]],
    name="causa",
    opacity=0.8,
    yaxis="y"
))
# Línea: IPC acumulado (eje derecho)
fig.add_trace(go.Bar(
    x=df["Año"],
    y=df[df.columns[2]],
    name="efecto",
))
fig.update_layout(
    title="Varaciones con k=1 / +0.85",
    xaxis=dict(title="Año", dtick=1,
               range=[1996.5,2023.5]),
    yaxis=dict(
        title="IPC anual (%)",
        range=[-1,1],
        side="left"),
    yaxis2=dict(
        title="IPC acumulado (%)",
        overlaying="y",
        #range=[.8,2.1],
        side="right"),
    height=500,
    barmode="group"
)
fig.show()

In [38]:
causa_14859 = pd.DataFrame(gastos_2023_wide[["Año","9.5-Educación no atribuible a ningún nivel", "5.1-Gestión de los residuos"]])
causa_14859.head()

,Año,9.5-Educación no atribuible a ningún nivel,5.1-Gestión de los residuos
0,1995,743.0,3663.0
1,1996,708.0,3757.0
2,1997,697.0,3994.0
3,1998,791.0,4236.0
4,1999,433.0,4450.0


In [39]:
df = causa_14859

fig = go.Figure()
# Barras: IPC anual (eje izquierdo)
fig.add_trace(go.Scatter(
    x=df["Año"],
    y=df[df.columns[1]],
    name="causa",
    mode="lines+markers",
    line=dict(width=3),
    yaxis="y"
))
# Línea: IPC acumulado (eje derecho)
fig.add_trace(go.Scatter(
    x=df["Año"],
    y=df[df.columns[2]],
    name="efecto",
    mode="lines+markers",
    line=dict(width=3),
    yaxis="y2"
))  
fig.update_layout(
    title="Datos con k=1 / +0.85",
    xaxis=dict(title="Año", dtick=1,
               range=[1996.5,2023.5]),
    yaxis=dict(
        title="IPC anual (%)",
        #range=[-1,1],
        side="left"),
    yaxis2=dict(
        title="IPC acumulado (%)",
        overlaying="y",
        #range=[.8,2.1],
        side="right"),
    height=500,
    barmode="group"
)
fig.show()

## Causa-Efecto 6824

In [40]:
causa_6824_var = pd.DataFrame(df_analisis_final_limpio[["9.3-Educación postsecundaria no terciaria_var", "7.2-Servicios ambulatorios_per_hab_var"]]).reset_index()
causa_6824_var.head()

,Año,9.3-Educación postsecundaria no terciaria_var,7.2-Servicios ambulatorios_per_hab_var
0,1997,-0.206609,-0.097726
1,1998,-0.206609,0.228269
2,1999,-0.231085,0.369481
3,2000,4.982379,0.624024
4,2001,-0.230783,-4.625133


In [41]:
df = causa_6824_var

fig = go.Figure()
# Barras: IPC anual (eje izquierdo)
fig.add_trace(go.Bar(
    x=df["Año"],
    y=df[df.columns[1]],
    name="causa",
    opacity=0.8,
    yaxis="y"
))
# Línea: IPC acumulado (eje derecho)
fig.add_trace(go.Bar(
    x=df["Año"],
    y=df[df.columns[2]],
    name="efecto",
))
fig.update_layout(
    title="Varaciones con k=1 / -0.926",
    xaxis=dict(title="Año", dtick=1,
               range=[1996.5,2023.5]),
    yaxis=dict(
        title="IPC anual (%)",
        range=[-1,1],
        side="left"),
    yaxis2=dict(
        title="IPC acumulado (%)",
        overlaying="y",
        #range=[.8,2.1],
        side="right"),
    height=500,
    barmode="group"
)
fig.show()

In [42]:
causa_6824_ef = pd.DataFrame(ratios_gastos_2023_wide[["7.2-Servicios ambulatorios_per_hab"]]).reset_index()
causa_6824_ef.head()



,Año,7.2-Servicios ambulatorios_per_hab
0,1995,885.868888
1,1996,900.920757
2,1997,891.801798
3,1998,914.211863
4,1999,951.144722


In [43]:
causa_6824_ca = pd.DataFrame(gastos_2023_wide[["Año","9.3-Educación postsecundaria no terciaria"]])
causa_6824_ca.head()

,Año,9.3-Educación postsecundaria no terciaria
0,1995,4.0
1,1996,4.0
2,1997,4.0
3,1998,4.0
4,1999,3.0


In [44]:
causa_6824_ca = causa_6824_ca.merge(causa_6824_ef, on="Año")
df = causa_6824_ca

fig = go.Figure()
# Barras: IPC anual (eje izquierdo)
fig.add_trace(go.Scatter(
    x=df["Año"],
    y=df[df.columns[1]],
    name="causa",
    mode="lines+markers",
    line=dict(width=3),
    yaxis="y"
))
# Línea: IPC acumulado (eje derecho)
fig.add_trace(go.Scatter(
    x=df["Año"],
    y=df[df.columns[2]],
    name="efecto",
    mode="lines+markers",
    line=dict(width=3),
    yaxis="y2"
))  
fig.update_layout(
    title="Datos con k=1 / -0.926",
    xaxis=dict(title="Año", dtick=1,
               range=[1996.5,2023.5]),
    yaxis=dict(
        title="IPC anual (%)",
        #range=[-1,1],
        side="left"),
    yaxis2=dict(
        title="IPC acumulado (%)",
        overlaying="y",
        #range=[.8,2.1],
        side="right"),
    height=500,
    barmode="group"
)
fig.show()